## Hiperparametros

Recordar ejecutar dentro de la carpeta notebooks

In [ ]:
df_clean = pd.read_csv("../TransManual.csv")
df_clean.head(5)

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,...,Transaccion_ID,Cliente_ID,Categoria,Metodo_de_pago,Locacion,Descuento_Aplicado,Precio_Unitario_Norm,Cantidad_Norm,Total_Norm,Timestamp_Norm
0,TXN_6867343,CUST_09,Patisserie,7,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,...,5159,8,7,2,1,1,0.375000,1.000000,0.444444,0.743935
1,TXN_3731986,CUST_22,Milk Products,62,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,...,2398,21,6,2,1,1,0.666667,0.888889,0.632099,0.510332
2,TXN_9303719,CUST_02,Butchers,17,21.5,2.0,43.0,Credit Card,Online,2022-10-05,...,7363,1,1,1,1,0,0.458333,0.111111,0.093827,0.248877
3,TXN_4575373,CUST_05,Food,172,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,...,3169,4,4,2,1,0,0.208333,0.666667,0.203704,0.246181
4,TXN_3652209,CUST_07,Food,84,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,...,2341,6,4,1,0,1,0.000000,0.777778,0.086420,0.471698


In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Opcion 1 - Uso de la Transformacion Manual

# Carga de datos
# iris = load_iris()
df_clean = pd.read_csv("../TransManual.csv")

# Columnas para usar
# 'Category',	'Item',	'Price Per Unit',	'Quantity',	'Total Spent',	'Payment Method',	'Location', 'Discount Applied'

X = df_clean.drop(columns=['Transaction ID',	'Customer ID',	'Category',	'Item',
                          'Price Per Unit',	'Quantity',	'Total Spent',	'Payment Method',	'Location',
                          'Transaction Date', 	'Discount Applied',	'Timestamp',
                          'Transaccion_ID', 'Cliente_ID', 'Timestamp_Norm', 'Metodo_de_pago'])
y = df_clean['Metodo_de_pago']

# División: 80% entrenamiento, 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
df_pipeline = pd.read_csv("../clean_retail_store_sales.csv")
df_pipeline.head(3)

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,Timestamp
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,1712534400
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,1690070400
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,1664928000


In [ ]:
# Opcion 2 - Uso de pipeline
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


df_pipeline = pd.read_csv("../clean_retail_store_sales.csv")
df_auto = df_pipeline.copy()

# Columnas para usar
# 'Category',	'Item',	'Price Per Unit',	'Quantity',	'Total Spent',	'Payment Method',	'Location', 'Discount Applied'

y = df_auto['Category']

X = df_auto.drop(columns=['Transaction ID',	'Customer ID', 'Category',
                          'Transaction Date', 'Timestamp'])

# División: 80% entrenamiento, 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(transformers=[
('cat', OneHotEncoder(), ['Item', 'Location', 'Discount Applied', 'Payment Method']),
('num', StandardScaler(), ["Price Per Unit",	"Quantity",	"Total Spent"])
])

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [ ]:
# Añadir linea solo para uso de Transformacion Manual
# modelo = RandomForestClassifier(random_state=42)

# Definir la grilla de hiperparámetros
# Usar SOLO para pipeline "classifier__" al inicio de cada indicador
# Ej : classifier__n_estimators:

param_grid = {
    'classifier__n_estimators': [10, 50, 100],      # Cantidad de árboles
    'classifier__max_depth': [None, 3, 5, 10],      # Profundidad de los árboles
    'classifier__criterion': ['gini', 'entropy']    # Medida de calidad de división
}

# Configurar la búsqueda con Validación Cruzada (CV=5)
# Se usa cv = 5, porque es recomendado, pero se puede ir probando con mas valores
# El 5 , no proviene de un calculo o grafico como el "codo"

# Sscoring = Define bajo que metrica guiarse para "ganar"
# Cambiar estimator segun uso de Pipeline o Manual
# Manual = modelo
# Pipeline = full_pipeline
grid_search = GridSearchCV(
      estimator=full_pipeline,
      param_grid=param_grid,
      cv=5,
      scoring='accuracy',
      n_jobs=-1,          # SOLO USAR EN COLAB, esto te usa todos los nucleos para calcular
      verbose=1
    )

# Ejecutar la optimización
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('cat',
                                                                         OneHotEncoder(),
                                                                         ['Item',
                                                                          'Location',
                                                                          'Discount '
                                                                          'Applied',
                                                                          'Payment '
                                                                          'Method']),
                                                                        ('num',
                                                                         StandardScaler(),
                                                                         ['Price '
                                                                          'Per '
                                                                          'Unit',
                                                                          'Quantity',
                                                                          'Total '
                                                                          'Spent'])])),
                                       ('classifier',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__criterion': ['gini', 'entropy'],
                         'classifier__max_depth': [None, 3, 5, 10],
                         'classifier__n_estimators': [10, 50, 100]},
             scoring='accuracy', verbose=1)

In [ ]:
print(f"Mejores Hiperparámetros: {grid_search.best_params_}")
print(f"Mejor Precisión (Accuracy) en entrenamiento: {grid_search.best_score_:.4f}")

# Evaluación final en el set de prueba
y_pred = grid_search.predict(X_test)
print("\nReporte de Clasificación Final:")
print(classification_report(y_test, y_pred))

Mejores Hiperparámetros: {'classifier__criterion': 'gini', 'classifier__max_depth': None, 'classifier__n_estimators': 100}
Mejor Precisión (Accuracy) en entrenamiento: 0.9984

Reporte de Clasificación Final:
                                    precision    recall  f1-score   support

                         Beverages       1.00      1.00      1.00       188
                          Butchers       1.00      1.00      1.00       193
Computers and electric accessories       1.00      1.00      1.00       191
     Electric household essentials       1.00      1.00      1.00       205
                              Food       1.00      1.00      1.00       204
                         Furniture       1.00      1.00      1.00       191
                     Milk Products       1.00      1.00      1.00       231
                        Patisserie       1.00      1.00      1.00       194

                          accuracy                           1.00      1597
                         macro